# 🎧 Qwen3-TTS Audiobook Studio — Kaggle Edition v4 (1.7B CustomVoice)

### Turn any text into professional audiobook audio

---

## How to Use
1. Run Cell 1 (Setup) — installs packages, ~2 min
2. Run Cell 2 (Helpers) — loads utilities & 1.7B model
3. Run Cell 3 (Launch) — starts Gradio & prints public URL

> 🔗 Link valid 72 hrs. First run downloads ~10 GB. Do NOT refresh during generation.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 1: SETUP & INSTALLATION
# ════════════════════════════════════════════════════════════════════
import sys, os, warnings
warnings.filterwarnings("ignore")
print("🔧 Setting up environment...")
!pip uninstall -y accelerate transformers qwen-tts 2>/dev/null
!pip install -q accelerate transformers
!pip install -q qwen-tts==0.1.1 soundfile deepmultilingualpunctuation pyloudnorm torchaudio "gradio>=4.44.0" safetensors
import torch, shutil
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if shutil.which("ffmpeg"): print("✅ ffmpeg found")
else: print("⚠️ ffmpeg not found")
print("✨ Setup complete!")

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 2: HELPERS (1.7B CustomVoice + Fixed Indentation)
# ════════════════════════════════════════════════════════════════════
import os, re, gc, time, json, shutil, zipfile, subprocess
from pathlib import Path
import torch, numpy as np, soundfile as sf
KAGGLE_INPUT, KAGGLE_WORKING = "/kaggle/input", "/kaggle/working"
CHECKPOINT_DIR = os.path.join(KAGGLE_WORKING, "checkpoints")
OUTPUT_DIR = KAGGLE_WORKING
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
SAMPLE_RATE = 24_000
model, punct_model = None, None
_stop_flag = [False]
def request_stop(): _stop_flag[0] = True
def clear_stop(): _stop_flag[0] = False
def is_stop_requested(): return _stop_flag[0]
def gpu_mem_str():
    if not torch.cuda.is_available(): return "CPU"
    used = torch.cuda.memory_reserved(0)/1024**3
    total = torch.cuda.get_device_properties(0).total_memory/1024**3
    return f"VRAM {used:.1f}/{total:.1f} GB"
if not torch.cuda.is_available(): print("⚠️ No GPU. Enable T4/P100.")
else: print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
try:
    import urllib.request; urllib.request.urlopen("https://huggingface.co", timeout=8)
    print("✅ Internet: OK")
except: print("⚠️ Internet: OFF")
VOICE_PRESETS = {
    "Narrator (Default)": "Speak in a calm, clear, measured voice.",
    "Documentary": "Confident, authoritative, deliberate pacing.",
    "Dramatic": "Emotional depth, expressive dynamics, strategic pauses.",
    "Conversational": "Warm, relaxed, natural conversation.",
    "News Anchor": "Clear, professional, crisp articulation.",
    "Custom": ""
}
def strip_markdown(t):
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.MULTILINE)
    t = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', t)
    t = re.sub(r'`{1,3}(.+?)`{1,3}', r'\1', t, flags=re.DOTALL)
    t = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', t)
    t = re.sub(r'^[\*\-\+]\s+', '', t, flags=re.MULTILINE)
    t = re.sub(r'^\d+\.\s+', '', t, flags=re.MULTILINE)
    return t.strip()
def flesch_kincaid_grade(t):
    s = max(1, len(re.findall(r'[.!?]+', t)))
    w = max(1, len(t.split()))
    y = max(1, len(re.findall(r'[aeiouyAEIOUY]+', t)))
    return 0.39*(w/s) + 11.8*(y/w) - 15.59
def analyse_text(t):
    t = t.strip()
    w = len(t.split())
    s = len(re.findall(r'[.!?]+', t))
    c = re.findall(r'^#{1,3}\s+(.+)$', t, re.MULTILINE)
    return {"words":w, "sentences":s, "chapters":c, "duration_min":w/155, "size_mb_est":(w/155)*2.88, "fk_grade":flesch_kincaid_grade(t) if w>20 else 0.0}
def smart_chunk_text(text, max_words=135):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, cur, cur_n = [], [], 0
    for para in paragraphs:
        for sent in re.split(r"(?<=[.!?])\s+", para):
            sw = sent.split()
            if not sw: continue
            if len(sw) > max_words:
                if cur: chunks.append(" ".join(cur)); cur = []; cur_n = 0
                mid = len(sw)//2; best = max_words
                for idx in range(max(0,mid-40), min(len(sw),mid+40)):
                    if sw[idx].endswith((".", "!", "?")): best = idx+1; break
                chunks.append(" ".join(sw[:best]))
                cur = sw[best:]; cur_n = len(cur)
            elif cur_n + len(sw) > max_words:
                if cur: chunks.append(" ".join(cur))
                cur = sw; cur_n = len(sw)
            else: cur.extend(sw); cur_n += len(sw)
    if cur: chunks.append(" ".join(cur))
    changed = True
    while changed:
        changed = False; merged = []; i = 0
        while i < len(chunks):
            c = chunks[i]
            if len(c.split()) < 25 and i < len(chunks)-1:
                c = c + " " + chunks[i+1]; i += 1; changed = True
            merged.append(c); i += 1
        chunks = merged
    return chunks
def detect_header(t):
    for line in t.strip().split("\n")[:3]:
        line = line.strip()
        if line.startswith("## ") or re.match(r"^#{1,3}\s", line):
            h = re.sub(r"^#+\s*", "", line).strip()
            return re.sub(r"[#*`]", "", h)[:80] or None
    return None
def safe_preview(t, max_chars=80):
    if len(t) <= max_chars: return t
    return t[:max_chars].rsplit(" ", 1)[0] + "\u2026"
def to_numpy(audio):
    if isinstance(audio, torch.Tensor): audio = audio.detach().cpu().float().numpy()
    return np.asarray(audio, dtype=np.float32).flatten()
def apply_speed(audio, speed, sr=SAMPLE_RATE):
    if abs(speed-1.0)<0.01: return audio
    import torchaudio
    return torchaudio.functional.resample(torch.from_numpy(audio).unsqueeze(0), int(sr*speed), sr).squeeze(0).numpy()
def normalize_audio(audio, mode="podcast", sr=SAMPLE_RATE):
    if mode in ("off", None): return audio
    audio = audio.astype(np.float32)
    if mode == "peak":
        peak = np.max(np.abs(audio))
        return audio*(0.95/peak) if peak>0 else audio
    try:
        import pyloudnorm as pyln
        meter = pyln.Meter(sr, block_size=0.400)
        a2 = audio[:, np.newaxis] if audio.ndim==1 else audio
        loud = meter.integrated_loudness(a2)
        targets = {"acx":-16.0, "podcast":-19.0, "youtube":-14.0}
        try: target = float(mode)
        except ValueError: target = targets.get(mode.lower(), -19.0)
        if loud>-70 and not np.isnan(loud):
            return pyln.normalize.loudness(a2, loud, target).flatten()
    except: pass
    peak = np.max(np.abs(audio))
    return audio*(0.95/peak) if peak>0 else audio
def fmt_time(s):
    h = int(s//3600); m = int((s%3600)//60); sec = int(s%60)
    return f"{h}:{m:02d}:{sec:02d}"
FFMPEG = shutil.which("ffmpeg")
def export_mp3(wav_path, mp3_path, bitrate="192k"):
    if not FFMPEG: return False
    try:
        r = subprocess.run([FFMPEG,"-y","-i",wav_path,"-codec:a","libmp3lame","-b:a",bitrate,mp3_path], capture_output=True, timeout=300)
        return r.returncode==0 and os.path.exists(mp3_path)
    except: return False
def split_into_chapters(merged, timestamps, header_indices, output_dir):
    if not timestamps: return []
    ch_idx = sorted(i for i in header_indices if i < len(timestamps))
    if len(ch_idx)<2: ch_idx = list(range(0, len(timestamps), max(1, len(timestamps)//20)))
    if ch_idx and ch_idx[0]!=0: ch_idx.insert(0, 0)
    cdir = os.path.join(output_dir, "chapters"); os.makedirs(cdir, exist_ok=True)
    saved = []
    for ci, si in enumerate(ch_idx):
        ei = ch_idx[ci+1] if ci+1 < len(ch_idx) else len(timestamps)
        ts, te = timestamps[si][0], timestamps[ei-1][1] if ei>0 else timestamps[-1][1]
        title = timestamps[si][2][:50].replace("/","-").replace("\\","-")
        fname = f"{ci+1:02d}_{title}.wav"; fpath = os.path.join(cdir, fname)
        ss, es = int(ts*SAMPLE_RATE), int(te*SAMPLE_RATE)
        chunk = merged[ss:es]
        if len(chunk)>0:
            sf.write(fpath, chunk, SAMPLE_RATE, subtype="PCM_24")
            saved.append((fname, ts, te, timestamps[si][2]))
    return saved
def package_outputs(files, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            if f and os.path.exists(f): zf.write(f, os.path.basename(f))
    return zip_path
def save_checkpoint(chunk_idx, new_audio, timestamps, metadata):
    sf.write(os.path.join(CHECKPOINT_DIR, f"chunk_{chunk_idx:05d}.wav"), new_audio, SAMPLE_RATE)
    meta = dict(metadata, last_chunk=chunk_idx, timestamp=time.time())
    for n, d in {"meta.json":meta, "timestamps.json":timestamps}.items():
        with open(os.path.join(CHECKPOINT_DIR, n), "w") as f: json.dump(d, f)
def load_latest_checkpoint():
    mf = os.path.join(CHECKPOINT_DIR, "meta.json")
    if not os.path.exists(mf): return None
    with open(mf) as f: metadata = json.load(f)
    tf = os.path.join(CHECKPOINT_DIR, "timestamps.json")
    timestamps = json.load(open(tf)) if os.path.exists(tf) else []
    segs = [sf.read(str(cf))[0].astype(np.float32) for cf in sorted(Path(CHECKPOINT_DIR).glob("chunk_*.wav"))]
    ct = timestamps[-1][1] if timestamps else 0.0
    return {"metadata":metadata, "audio_segments":segs, "timestamps":timestamps, "last_chunk":metadata.get("last_chunk",0), "current_time":ct}
def clear_checkpoints():
    if os.path.exists(CHECKPOINT_DIR): shutil.rmtree(CHECKPOINT_DIR)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("✅ Helpers loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 3: GRADIO UI & LAUNCH
# ════════════════════════════════════════════════════════════════════
import gradio as gr, threading
def run_generation(file_obj, text_paste, voice_choice, preset_name, custom_style, speed,
                   norm_mode, para_pause, chap_pause, do_mp3, do_chapters, do_resume,
                   sample_only=False, progress=gr.Progress()):
    clear_stop()
    vs = custom_style.strip() if preset_name=="Custom" else VOICE_PRESETS.get(preset_name, "")
    vn = voice_choice.split(" (")[0]
    log = []
    def emit(s="", a="", aud=None, wav=None, mp3=None, zipf=None, cmd=""):
        if a: log.append(a)
        return s, "\n".join(log[-30:]), aud, wav, mp3, zipf, cmd
    def pbar(pct, lbl=""):
        return f'<div style="font-family:system-ui;padding:16px;background:#fff;border:1px solid #E5E7EB;border-radius:12px"><div style="font-size:36px;font-weight:800;color:#6366F1;text-align:center">{pct:.0f}%</div><div style="background:#E5E7EB;border-radius:8px;height:16px;overflow:hidden;margin:10px 0"><div style="width:{pct:.0f}%;height:100%;background:linear-gradient(90deg,#4F46E5,#7C3AED,#9333EA);border-radius:8px;transition:width .4s"></div></div><div style="text-align:center;color:#6B7280;font-size:14px">{lbl}</div></div>'
    script = ""
    if file_obj is not None:
        try:
            p = file_obj if isinstance(file_obj, str) else file_obj.name
            with open(p, "r", encoding="utf-8", errors="replace") as fh: script = fh.read()
        except Exception as e: yield emit("", f"❌ File read error: {e}"); return
    if not script.strip(): script = text_paste.strip()
    if not script: yield emit("", "❌ No text provided."); return
    if model is None:
        log.append("⏳ Loading models (~10 GB, 5–30 min)...")
        yield emit(pbar(0, "Loading models... DO NOT refresh."), log[-1])
        res, exc = [None], [None]
        def _load():
            try: res[0] = load_models(log=log.append)
            except Exception as e: exc[0] = e; res[0] = False
        t = threading.Thread(target=_load, daemon=True); t.start()
        while t.is_alive():
            t.join(timeout=2.5)
            yield emit(pbar(0, f"Loading… {gpu_mem_str()}"), log[-1] if log else "⏳ Downloading weights…")
        if exc[0] or not res[0]:
            yield emit("", f"❌ Load failed: {exc[0] or 'unknown'}"); return
        log.append("✅ Models loaded.")
    log.append("📝 Pre-processing..."); yield emit(pbar(1, "Pre-processing..."), log[-1])
    script = strip_markdown(script)
    script = punct_model.restore_punctuation(script)
    chunks = smart_chunk_text(script)
    if sample_only: chunks = chunks[:2]
    total = len(chunks)
    log.append(f"✂️ {total} chunks"); yield emit(pbar(2, f"{total} chunks"), log[-1])
    sc, segs, ts, ct, fail, hdr = 0, [], [], 0.0, [], set()
    if do_resume and not sample_only:
        ckpt = load_latest_checkpoint()
        if ckpt:
            sc, segs, ts, ct = ckpt["last_chunk"]+1, ckpt["audio_segments"], ckpt["timestamps"], ckpt["current_time"]
            log.append(f"🔁 Resuming from {sc}"); yield emit(pbar(0, f"Resumed at {sc}"), log[-1])
    ema, t0 = None, time.time()
    for i in range(sc, total):
        if is_stop_requested(): log.append("🛑 Stopped."); yield emit(pbar((i/total)*100, "Stopped"), log[-1]); break
        ch = chunks[i]; pct = (i+1)/total*100
        is_h = detect_header(ch) is not None
        if is_h: hdr.add(i)
        pd = chap_pause if is_h else para_pause
        d = i-sc; el = time.time()-t0
        eta = (ema*(total-i-1)) if ema and d>0 else (el/d*(total-i-1) if d>0 else 0)
        lbl = f"Chunk {i+1}/{total} · ETA {fmt_time(eta)} · {gpu_mem_str()}"
        log.append(f" [{i+1}/{total}] {safe_preview(ch, 50)}")
        progress(pct/100, desc=f"Chunk {i+1}/{total}")
        yield emit(pbar(pct, lbl), log[-1])
        out_np, ok = None, False
        subs = [ch]; si = 0
        while si < len(subs):
            sub = subs[si]; si += 1
            if is_stop_requested(): break
            for att in range(3):
                if is_stop_requested(): break
                try:
                    t1 = time.time()
                    w, sr = model.generate_custom_voice(text=sub, language="English", speaker=vn, instruct=vs)
                    out_np = to_numpy(w[0] if isinstance(w,(list,tuple)) else w)
                    if len(out_np)>0:
                        if abs(speed-1.0)>0.01: out_np = apply_speed(out_np, speed)
                        ok = True; dur = time.time()-t1
                        ema = dur if ema is None else 0.7*ema+0.3*dur
                        break
                except RuntimeError as e:
                    if "out of memory" in str(e).lower() and len(sub.split())>30:
                        torch.cuda.empty_cache(); gc.collect()
                        ws = sub.split(); mid = len(ws)//2
                        subs.insert(si, " ".join(ws[mid:]))
                        subs.insert(si, " ".join(ws[:mid]))
                        break
                except: pass
                if att<2: time.sleep((att+1)*1.5)
            if ok: break
        if not ok or out_np is None:
            fail.append(i+1)
            out_np = np.zeros(int(len(ch.split())/155*SAMPLE_RATE), dtype=np.float32)
        sd = len(out_np)/SAMPLE_RATE
        segs.append(out_np)
        if i<total-1 and pd>0:
            segs.append(np.zeros(int(pd*SAMPLE_RATE), dtype=np.float32))
            sd += pd
        tss = ct; ct += sd
        ttl = detect_header(ch) or safe_preview(re.split(r"(?<=[.!?])\s+", ch, maxsplit=1)[0], 50)
        ts.append((tss, ct, ttl))
        if not sample_only and (i+1)%5==0: save_checkpoint(i, out_np, ts, {"total_chunks":total, "voice":vn, "failed":fail})
        if (i+1)%10==0 and torch.cuda.is_available(): torch.cuda.empty_cache(); gc.collect()
    if not segs: yield emit("", "❌ No audio."); return
    yield emit(pbar(99, "Merging..."), "🔗 Merging...")
    merged = np.concatenate(segs); del segs; gc.collect()
    if norm_mode != "off": merged = normalize_audio(merged, mode=norm_mode)
    dur = len(merged)/SAMPLE_RATE
    suf = "_sample" if sample_only else "_output"
    wp = os.path.join(OUTPUT_DIR, f"audiobook{suf}.wav")
    sf.write(wp, merged, SAMPLE_RATE, subtype="PCM_24")
    fmb = os.path.getsize(wp)/1024**2
    pp = os.path.join(OUTPUT_DIR, f"audiobook{suf}_preview.wav")
    sf.write(pp, merged[:min(60*SAMPLE_RATE, len(merged))], SAMPLE_RATE, subtype="PCM_16")
    cp = os.path.join(OUTPUT_DIR, "audiobook_chapters.txt")
    with open(cp, "w", encoding="utf-8") as f:
        f.write("# Chapter Markers\n# Format: H:MM:SS - H:MM:SS | Title\n\n")
        for s,e,t in ts: f.write(f"{fmt_time(s)} - {fmt_time(e)} | {t}\n")
    cstr = "### Chapter Markers\n\n| Start | End | Title |\n|---|---|---|\n"
    for s,e,t in ts: cstr += f"| {fmt_time(s)} | {fmt_time(e)} | {t} |\n"
    olist = [wp, cp]
    if do_chapters and not sample_only:
        scs = split_into_chapters(merged, ts, hdr, OUTPUT_DIR)
        olist.extend([os.path.join(OUTPUT_DIR, "chapters", fn) for fn, _, _, _ in scs])
    mp3p = None
    if do_mp3 and not sample_only:
        mp3p = os.path.join(OUTPUT_DIR, "audiobook_output.mp3")
        if export_mp3(wp, mp3p): olist.append(mp3p)
        else: mp3p = None; log.append("⚠️ MP3 failed.")
    zp = os.path.join(OUTPUT_DIR, "audiobook_all_files.zip") if not sample_only else None
    if zp: package_outputs(olist, zp)
    log.append(f"✅ Done! {fmt_time(dur)} · {fmb:.1f} MB · {vn}")
    dh = f'<div style="background:linear-gradient(135deg,#059669,#10B981);color:white;border-radius:12px;padding:24px;text-align:center"><div style="font-size:28px;font-weight:800">{"🎤 Sample Ready!" if sample_only else "🎉 Audiobook Ready!"}</div><div style="margin-top:8px;font-size:15px">{fmt_time(dur)} · {fmb:.1f} MB · {vn}</div></div>'
    yield emit(dh, log[-1], pp, wp, mp3p, zp, cstr)
CSS = ".gradio-container { max-width: 1100px !important; } footer { display: none !important; }"
THEME = gr.themes.Soft(primary_hue=gr.themes.colors.indigo, secondary_hue=gr.themes.colors.purple, neutral_hue=gr.themes.colors.slate)
with gr.Blocks(theme=THEME, css=CSS, title="🎧 Audiobook Studio") as demo:
    gr.HTML('<div style="background:linear-gradient(135deg,#4F46E5,#7C3AED,#9333EA);color:white;border-radius:16px;padding:28px;text-align:center;margin-bottom:8px"><div style="font-size:32px;font-weight:800">🎧 Audiobook Studio</div><div style="font-size:15px;opacity:.9">Qwen3-TTS 1.7B · Kaggle v4</div></div>')
    with gr.Tabs():
        with gr.Tab("🎙️ Studio"):
            with gr.Row():
                with gr.Column(scale=5):
                    gr.Markdown("### 📄 Script")
                    fi = gr.File(label=".txt", file_types=[".txt"])
                    ti = gr.Textbox(label="Paste text", placeholder="## Chapter 1\n...", lines=10)
                    sb = gr.HTML('<div style="color:#9CA3AF;font-size:13px">Stats appear here.</div>')
                with gr.Column(scale=4):
                    gr.Markdown("### 🎙️ Voice")
                    vr = gr.Radio(["Ryan (Warm)", "Aiden (Deep)"], value="Ryan (Warm)", label="Voice")
                    pd = gr.Dropdown(list(VOICE_PRESETS.keys()), value="Narrator (Default)", label="Preset")
                    cs = gr.Textbox(label="Style", value=VOICE_PRESETS["Narrator (Default)"], lines=2)
                    ss = gr.Slider(0.7, 1.4, value=1.0, step=0.05, label="Speed")
                    gr.Markdown("### ⚙️ Quality")
                    nd = gr.Dropdown([("Podcast", "podcast"), ("ACX", "acx"), ("YouTube", "youtube"), ("Peak", "peak"), ("Off", "off")], value="podcast", label="Normalize")
                    ps = gr.Slider(0, 2, 0.6, 0.1, label="Para pause")
                    chs = gr.Slider(0.3, 3, 1.2, 0.1, label="Chapter pause")
                    gr.Markdown("### 📦 Export")
                    mc = gr.Checkbox("MP3"); cc = gr.Checkbox("Chapters"); rc = gr.Checkbox("Resume")
                    with gr.Row():
                        sm = gr.Button("🎤 Sample", scale=2); gb = gr.Button("🚀 Generate", variant="primary", scale=3); st = gr.Button("🛑 Stop", scale=1)
                    sh = gr.HTML('<div style="color:#9CA3AF">Ready.</div>')
                    lb = gr.Textbox(label="Log", lines=6, interactive=False)
        with gr.Tab("🎧 Results"):
            ap = gr.Audio(label="Preview")
            wd = gr.File("WAV"); md = gr.File("MP3"); zd = gr.File("ZIP")
            cm = gr.Markdown("*Generate to see chapters.*")
        with gr.Tab("⚙️ Advanced"):
            gr.Markdown("Save Version → Quick Save → Download from /kaggle/working/")
            cb = gr.Button("🗑️ Clear Checkpoints"); cst = gr.Textbox("Status")
    GO = [sh, lb, ap, wd, md, zd, cm]
    def mfn(s):
        def f(*a, progress=gr.Progress()): yield from run_generation(*a, sample_only=s, progress=progress)
        return f
    IN = [fi, ti, vr, pd, cs, ss, nd, ps, chs, mc, cc, rc]
    gb.click(mfn(False), IN, GO); sm.click(mfn(True), IN, GO); st.click(request_stop)
    pd.change(lambda p: VOICE_PRESETS.get(p, ""), [pd], [cs])
    def on_t(t):
        if not t.strip(): return '<div style="color:#9CA3AF">Stats...</div>'
        s = analyse_text(t)
        fk = "Easy" if s["fk_grade"]<6 else "Medium" if s["fk_grade"]<10 else "Hard" if s["fk_grade"]<14 else "Very Hard"
        p = 'background:#EEF2FF;color:#4F46E5;border-radius:6px;padding:4px 8px;font-size:12px;'
        return f'<div style="display:flex;flex-wrap:wrap;gap:6px"><span style="{p}">{s["words"]:,} w</span><span style="{p}">~{s["duration_min"]:.0f}m</span><span style="{p}">{fk}</span></div>'
    ti.change(on_t, [ti], [sb])
    def on_f(f):
        if not f: return "", '<div style="color:#9CA3AF">Stats...</div>'
        try:
            p = f if isinstance(f, str) else f.name
            with open(p, "r", encoding="utf-8", errors="replace") as fh: c = fh.read()
            return c, on_t(c)
        except Exception as e: return "", f'<div style="color:#DC2626">❌ {e}</div>'
    fi.change(on_f, [fi], [ti, sb])
    cb.click(lambda: (clear_checkpoints(), "✅ Cleared."), outputs=[cst])
print("🚀 Launching...")
demo.queue(max_size=1).launch(share=True, server_port=7860, show_error=True)